# HIFA — Train trên Google Colab (GPU)

Notebook chạy các kiến trúc có LLM (A1 FLAG-Trader, B1 HARLF, C1 Debate, HIFA) trên GPU Colab.
Trước khi chạy: **Runtime → Change runtime type → GPU (T4/A100)**.

Xem hướng dẫn chi tiết ở `COLAB_GUIDE.md`.

## 1. Kiểm tra GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'KHÔNG thấy GPU! Vào Runtime → Change runtime type → GPU rồi chạy lại.'
print('CUDA available:', torch.cuda.is_available())
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Lấy code
Sửa URL repo của bạn (Cách A - GitHub). Nếu dùng Drive, xem COLAB_GUIDE.md mục 1 Ô 2.

In [ ]:
!git clone https://github.com/<username>/hifa.git
%cd hifa
# Set PYTHONPATH một lần cho MỌI cell `!python` phía dưới (an toàn hơn gán inline
# kiểu `!PYTHONPATH=. python ...` vốn không đáng tin trong magic `!` của Colab).
%env PYTHONPATH=/content/hifa

## 3. Cài dependencies (KHÔNG cài lại torch)
Chỉ báo kỹ thuật đã tự implement bằng pandas thuần (`src/data/build_indicators.py`),
nên KHÔNG cần `pandas-ta` nữa → dùng pandas/numpy bản mới nhất của Colab, không phải ghim version.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes \
    gymnasium stable-baselines3 yfinance quantstats \
    pyyaml scipy tensorboard pyarrow
import pandas, numpy
print('pandas:', pandas.__version__, '| numpy:', numpy.__version__)

## 4. Lấy dữ liệu Mỹ (fetch qua yfinance, ~2 phút)

In [ ]:
!python -m src.data.fetch_ohlcv_us --phase 0 --benchmark both
!python -m src.data.build_dataset --phase 0

## 5. Smoke test LLM thật trên GPU (verify trước khi train dài)

In [ ]:
!python scripts/_smoke_test_llm_real.py

## 6. Train A1 (FLAG-Trader-style), 5 seed
`PortfolioPolicyLLM` tự phát hiện GPU (`torch.cuda.is_available()`). Checkpoint lưu ở `checkpoints/`.

In [ ]:
for seed in range(5):
    print(f'=== Training A1 seed {seed} ===')
    !python -m src.training.train_phase_a --config configs/phase_a.yaml --seed {seed}

## 7. Baseline tài chính (B1 Index / B2 / B3 / B4 PPO) trên data Mỹ

In [ ]:
!python scripts/_run_baselines_us_phase0.py

## 8. Theo dõi training bằng TensorBoard (tùy chọn)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 9. Lưu kết quả về Google Drive (QUAN TRỌNG)
Colab xóa toàn bộ file khi hết phiên — luôn copy checkpoints/logs/reports về Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/LUAN_VAN/hifa_results"
!cp -r checkpoints logs reports "/content/drive/MyDrive/LUAN_VAN/hifa_results/"
print('Da luu checkpoints + logs + reports ve Drive.')